In [12]:
!which python

/home/user/jfayzullaev/stellar-clustering/.venv-vis/bin/python


In [13]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    normalized_mutual_info_score as NMI,
    adjusted_rand_score as ARI,
    adjusted_mutual_info_score as AMI,
    fowlkes_mallows_score as FMI,
    homogeneity_completeness_v_measure
)

In [14]:
RES_FILE = "../louvain_result_res0.5.csv"



NORM_LABELS_PATH = os.path.expanduser("~/stellar-clustering/publication/labeled-data/normalization/labels_mapped_normalized.csv")


In [15]:
N_SPLITS = 5
RANDOM_STATE = 42

In [16]:
def overall_purity(df_comm_name: pd.DataFrame) -> float:
    if df_comm_name.empty:
        return np.nan
    counts = df_comm_name.groupby(['community', 'name']).size().reset_index(name='cnt')
    totals = counts.groupby('community')['cnt'].sum()
    max_per_comm = counts.groupby('community')['cnt'].max()
    return float(max_per_comm.sum() / totals.sum())

In [ ]:

def load_communities_fixed_resolution(res_file: str) -> pd.DataFrame:
    
    df = pd.read_csv(res_file)

    if 'account_id' not in df.columns and 'node' in df.columns:
        df = df.rename(columns={'node': 'account_id'})
        
    df = df[['account_id', 'community']].dropna().drop_duplicates()

    try:
        df['account_id'] = df['account_id'].astype(int)
    except Exception:
        df['account_id'] = df['account_id'].astype(str)
    return df

In [ ]:
def evaluate_cv(
    labels_path: str,
    comm_df: pd.DataFrame,
    label_col: str = "name",
    n_splits: int = 5,
    random_state: int = 42
):


    labels = (pd.read_csv(labels_path)
                .dropna(subset=['account_id', label_col])
                .drop_duplicates(subset=['account_id'])
                .rename(columns={label_col: 'name'}))

    try:
        labels['account_id'] = labels['account_id'].astype(int)
        comm_cast = comm_df.copy()
        comm_cast['account_id'] = comm_cast['account_id'].astype(int)
    except Exception:
        labels['account_id'] = labels['account_id'].astype(str)
        comm_cast = comm_df.copy()
        comm_cast['account_id'] = comm_cast['account_id'].astype(str)


    joined = labels.merge(comm_cast, on='account_id', how='inner')

    n_labeled = len(labels)
    n_joined  = len(joined)
    coverage  = (n_joined / n_labeled) if n_labeled else 0.0


    le = LabelEncoder()
    y_all = le.fit_transform(joined['name'].values)
    X_ids = joined['account_id'].values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_ids, y_all), start=1):
        df_tr = joined.iloc[tr_idx].copy()
        df_te = joined.iloc[te_idx].copy()

 
        y_true_tr = le.transform(df_tr['name'])
        y_pred_tr = df_tr['community'].values
        
        nmi_tr = NMI(y_true_tr, y_pred_tr)
        ari_tr = ARI(y_true_tr, y_pred_tr)
        ami_tr = AMI(y_true_tr, y_pred_tr)
        fmi_tr = FMI(y_true_tr, y_pred_tr)
        homo_tr, comp_tr, v_tr = homogeneity_completeness_v_measure(y_true_tr, y_pred_tr)
        purity_tr = overall_purity(df_tr[['community', 'name']])


        y_true_te = le.transform(df_te['name'])
        y_pred_te = df_te['community'].values
        
        nmi_te = NMI(y_true_te, y_pred_te)
        ari_te = ARI(y_true_te, y_pred_te)
        ami_te = AMI(y_true_te, y_pred_te)
        fmi_te = FMI(y_true_te, y_pred_te)
        homo_te, comp_te, v_te = homogeneity_completeness_v_measure(y_true_te, y_pred_te)
        purity_te = overall_purity(df_te[['community', 'name']])

        rows.append({
            'fold': fold,
            'n_train': len(df_tr),
            'n_test': len(df_te),
            'train_frac': len(df_tr) / len(joined),

   
            'NMI_train': nmi_tr,
            'ARI_train': ari_tr,
            'AMI_train': ami_tr,
            'FMI_train': fmi_tr,
            'Homogeneity_train': homo_tr,
            'Completeness_train': comp_tr,
            'V-measure_train': v_tr,
            'Purity_train': purity_tr,


            'NMI_test': nmi_te,
            'ARI_test': ari_te,
            'AMI_test': ami_te,
            'FMI_test': fmi_te,
            'Homogeneity_test': homo_te,
            'Completeness_test': comp_te,
            'V-measure_test': v_te,
            'Purity_test': purity_te,
        })

    per_fold_df = pd.DataFrame(rows)


    metric_names = ['NMI', 'ARI', 'AMI', 'FMI', 'Homogeneity', 'Completeness', 'V-measure', 'Purity']
    averages = {}
    
    for metric in metric_names:
        averages[f'Avg_{metric}_train'] = float(np.nanmean(per_fold_df[f'{metric}_train'].values))
        averages[f'Avg_{metric}_test'] = float(np.nanmean(per_fold_df[f'{metric}_test'].values))
        averages[f'Std_{metric}_test'] = float(np.nanstd(per_fold_df[f'{metric}_test'].values))

    averages['Avg_train_frac'] = float(np.nanmean(per_fold_df['train_frac'].values))

    coverage_info = {
        'n_labeled': int(n_labeled),
        'n_joined': int(n_joined),
        'coverage': float(coverage),
    }

    return per_fold_df, averages, coverage_info

In [ ]:
comm_df = load_communities_fixed_resolution(RES_FILE)
print(f"Rows: {len(comm_df):,},  Unique accounts: {comm_df['account_id'].nunique():,}")


Loaded communities from: ../louvain_result_res0.5.csv
Rows: 4,315,652  |  Unique accounts: 4,315,652


### NORMALIZED LABELS

In [ ]:
norm_per_fold, norm_avg, norm_cov = evaluate_cv(
    labels_path=NORM_LABELS_PATH,
    comm_df=comm_df,
    label_col="name",
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE
)

print("NORMALIZED labels: per-fold metrics (TEST)")
display(norm_per_fold)

print("\nNORMALIZED labels: averages (TEST)")
for k, v in norm_avg.items():
    print(f"{k}: {v:.6f}")
print(f"Coverage: {norm_cov['coverage']:.2%}  ({norm_cov['n_joined']}/{norm_cov['n_labeled']})")


norm_per_fold.to_csv("cv_results_norm_per_fold.csv", index=False)
pd.DataFrame([{
    **norm_avg,
    **norm_cov
}]).to_csv("cv_results_norm_summary.csv", index=False)



NORMALIZED labels: per-fold metrics (TEST)


/home/user/jfayzullaev/stellar-clustering/.venv-vis/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


,fold,n_train,n_test,train_frac,NMI_train,ARI_train,AMI_train,FMI_train,Homogeneity_train,Completeness_train,V-measure_train,Purity_train,NMI_test,ARI_test,AMI_test,FMI_test,Homogeneity_test,Completeness_test,V-measure_test,Purity_test
0,1,6668,1668,0.799904,0.120011,0.044523,0.087059,0.677239,0.159316,0.096262,0.120011,0.935213,0.128825,0.049241,0.083738,0.685219,0.177830,0.100993,0.128825,0.935252
1,2,6669,1667,0.800024,0.118962,0.046621,0.085533,0.681304,0.157704,0.095501,0.118962,0.935073,0.136565,0.042831,0.094615,0.670849,0.189988,0.106592,0.136565,0.935813
2,3,6669,1667,0.800024,0.119786,0.045229,0.087198,0.676147,0.160424,0.095575,0.119786,0.935223,0.134535,0.045956,0.093095,0.689313,0.181093,0.107020,0.134535,0.935813
3,4,6669,1667,0.800024,0.120763,0.043492,0.088423,0.676754,0.160243,0.096891,0.120763,0.935223,0.134615,0.054601,0.090234,0.687788,0.187744,0.104923,0.134615,0.935213
4,5,6669,1667,0.800024,0.122364,0.047486,0.090430,0.682441,0.161936,0.098334,0.122364,0.935373,0.122445,0.037106,0.075983,0.665025,0.172927,0.094777,0.122445,0.934613



NORMALIZED labels: averages (TEST)
Avg_NMI_train: 0.120377
Avg_NMI_test: 0.131397
Std_NMI_test: 0.005170
Avg_ARI_train: 0.045470
Avg_ARI_test: 0.045947
Std_ARI_test: 0.005893
Avg_AMI_train: 0.087729
Avg_AMI_test: 0.087533
Std_AMI_test: 0.006873
Avg_FMI_train: 0.678777
Avg_FMI_test: 0.679639
Std_FMI_test: 0.009818
Avg_Homogeneity_train: 0.159925
Avg_Homogeneity_test: 0.181917
Std_Homogeneity_test: 0.006282
Avg_Completeness_train: 0.096512
Avg_Completeness_test: 0.102861
Std_Completeness_test: 0.004567
Avg_V-measure_train: 0.120377
Avg_V-measure_test: 0.131397
Std_V-measure_test: 0.005170
Avg_Purity_train: 0.935221
Avg_Purity_test: 0.935341
Std_Purity_test: 0.000447
Avg_train_frac: 0.800000
Coverage: 100.00%  (8336/8336)

Saved: cv_results_norm_per_fold.csv, cv_results_norm_summary.csv


In [ ]:
summary_metrics = ['NMI', 'ARI', 'AMI', 'FMI', 'Homogeneity', 'Completeness', 'V-measure', 'Purity']
summary_data = []

for metric in summary_metrics:
    summary_data.append({
        'Metric': metric,
        'Mean': norm_avg[f'Avg_{metric}_test'],
        'Std': norm_avg[f'Std_{metric}_test']
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f"\nCoverage: {norm_cov['coverage']:.2%}  ({norm_cov['n_joined']}/{norm_cov['n_labeled']})")


norm_per_fold.to_csv("cv_results_norm_per_fold.csv", index=False)
pd.DataFrame([{**norm_avg, **norm_cov}]).to_csv("cv_results_norm_summary.csv", index=False)
summary_df.to_csv("cv_results_norm_summary_table.csv", index=False)


,Metric,Mean,Std
0,NMI,0.131397,0.005170
1,ARI,0.045947,0.005893
2,AMI,0.087533,0.006873
3,FMI,0.679639,0.009818
4,Homogeneity,0.181917,0.006282
5,Completeness,0.102861,0.004567
6,V-measure,0.131397,0.005170
7,Purity,0.935341,0.000447



Coverage: 100.00%  (8336/8336)
